In [ ]:
import pandas as pd
import numpy as np 
import plotly.express as px

In [ ]:
base = pd.read_csv("../data/intermin/data_daily.csv", index_col=0, parse_dates=["date_from", "date_to"])
cal  = pd.read_csv("../data/features/calendar/data_daily.csv", index_col=0)
df_day = base.join(cal)

base = pd.read_csv("../data/intermin/data_hourly.csv", index_col=0, parse_dates=["date_from", "date_to"])
cal  = pd.read_csv("../data/features/calendar/data_hourly.csv", index_col=0)
df_hour = base.join(cal)



In [ ]:
CUTOFF = '2024-01-01'
df_day_train = df_day[df_day['date_from'] < CUTOFF]
df_day_test = df_day[df_day['date_from'] >= CUTOFF]
df_hour_train = df_hour[df_hour['date_from'] < CUTOFF]
df_hour_test = df_hour[df_hour['date_from'] >= CUTOFF]

## Actual Consumption

### Daily Trend

In [ ]:
df_day_train = df_day_train.sort_values("date_from")
df_day_train["actual_consumption_ma7"] = df_day_train["actual_consumption"].rolling(window = 7).mean()

fig = px.line(df_day_train, 
              x = "date_from", 
              y = ["actual_consumption", "actual_consumption_ma7"],
              color_discrete_sequence=["#3D97CA", "#E69F00"],
              title = "Actual Consumption Daily Trend: 2019 - 2025")

fig.update_xaxes(title = "year")
fig.update_yaxes(title = "MWh/day")
fig.update_layout(
    legend=dict(
        orientation="h",      # horizontal
        yanchor="top",
        y=-0.2,               # below the plot area (negative = outside)
        xanchor="center",
        x=0.5,
        title_text="",        # remove the "variable" legend title
    ),
    title_x = 0.5
)
fig.show()

- **Observation**: The daily series swings between ≈ 0.87M (2024-03-31, Easter Sunday) and ≈ 1.68M MWh/day (2021-11-30). The 7-day average removes the weekly cycle and leaves a clear **annual wave**: highs in winter (up to ≈ 1.57M in Feb 2021), lows in summer (down to ≈ 1.10M in May 2023). The yearly maxima are lower in 2023 (≈ 1.57M) and 2024 (≈ 1.60M) than in 2019–2022 (≈ 1.65–1.68M).
- **Interpretation**: The wide spread of the raw line comes from the weekday/weekend cycle (see Weekday Effect) and from holidays; the 7-day average hides both, so it should not be used to judge single days. The lower winter peaks after 2022 are consistent with the 2022–2023 drop found later in the MSTL trend.
- **Implication**: Seasonality and the weekly cycle dominate the variance, so any trend or event comparison needs them removed first (see the by-month and STL sections).

### Mean consumption by month, per year

In [ ]:
df_day_train["date_from"] = pd.to_datetime(df_day_train["date_from"])
df_day_train["year"] = df_day_train["date_from"].dt.year

monthly = (
    df_day_train.groupby(["year", "month"], as_index=False)["actual_consumption"].mean()
)
monthly["year"] = monthly["year"].astype(str)  # categorical, so each year gets a distinct color

fig = px.line(
    monthly,
    x = "month",
    y = "actual_consumption",
    color = "year",
    markers = True,
    title = "Mean daily actual consumption by month, per year",
    labels = {"actual_consumption": "MWh/day", "month": "Month", "year": "Year"},
    color_discrete_sequence = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#CC79A7"],  # Okabe-Ito
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_layout(
    legend = dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5, title_text=""),
    title_x = 0.5
)
fig.show()


- **Observation**: Winter peaks (~1.5M MWh/day), summer troughs (~1.2M). 2020 drops in Mar–May, 2022 drops in H2, 2023 is the lowest year overall.
- **Interpretation**: 2020 matches the first lockdown; 2022 matches the energy crisis (high prices, saving). Weather is not controlled here.
- **Implication**: Month is a strong feature; year-level shifts are not purely seasonal.


### Weekday Effect

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df_day_train["date_from"] = pd.to_datetime(df_day_train["date_from"])
df_day_train["year"] = df_day_train["date_from"].dt.year

# 1. Remove level/season: ratio to centered 7-day mean
df_day_train["ratio_to_week"] = (
    df_day_train["actual_consumption"]
    / df_day_train["actual_consumption"].rolling(7, center=True).mean()
)

# 2. Day type: holidays on weekdays are their own group
def day_type(r):
    weekend = r["day_of_week_raw"] >= 5
    if r["is_holiday"]:
        return "Weekend holiday" if weekend else "Weekday holiday"
    return "Weekend" if weekend else "Weekday"

day_type_order = ["Weekday", "Weekday holiday", "Weekend", "Weekend holiday"]
df_day_train["day_type"] = pd.Categorical(
    df_day_train.apply(day_type, axis=1), categories=day_type_order, ordered=True
)


In [ ]:
# (a) 4 day types, pooled across years: box plot
fig = px.box(df_day_train.dropna(subset=["ratio_to_week"]),
             x="day_type", y="ratio_to_week", points="all",
             title="Day type effect (ratio to centered 7-day mean)")
fig.update_traces(marker=dict(size=3, opacity=0.5), jitter=0.4)
fig.update_layout(title_x = 0.5)
fig.show()

In [ ]:
# (b) Mon-Sun detail by year, normal days only
normal = df_day_train[df_day_train["day_type"].isin(["Weekday", "Weekend"])].copy()
normal["day_of_week"] = pd.Categorical(normal["day_of_week"], categories=order, ordered=True)
dow = (normal.groupby(["year", "day_of_week"], observed=True, as_index=False)["ratio_to_week"].mean()
             .sort_values(["year", "day_of_week"]))
dow["year"] = dow["year"].astype(str)
fig = px.line(dow, x="day_of_week", y="ratio_to_week", color="year", markers=True,
              title="Weekday effect, holidays excluded")
fig.update_layout(title_x = 0.5)
fig.show()

##
- **Observation**: Mon–Fri ≈ 1.04–1.07x of the weekly mean, Saturday ≈ 0.90x, Sunday ≈ 0.84x. The pattern is almost identical across 2019–2024.
- **Interpretation**: Economic is less active during weekends (people do not work). Monday and Friday have slightly lower consumption, possibly due to people working remotely on that two days.
- **Implication**: day_type (4 classes) is a useful feature; weekend holidays are NOT lower than regular weekends.

### Intra-day Pattern by Season

In [ ]:
df_hour_train["date_from"] = pd.to_datetime(df_hour_train["date_from"])

# 1. Drop DST switch days (23 or 25 hours)
day = df_hour_train["date_from"].dt.floor("D")
n_hours = df_hour_train.groupby(day)["date_from"].transform("size")
h = df_hour_train[n_hours == 24].copy()

# 2. Features
h["hour"] = h["date_from"].dt.hour

day_type_order = ["Weekday", "Weekday holiday", "Weekend", "Weekend holiday"]
h["day_type"] = pd.Categorical(h.apply(day_type, axis=1), categories=day_type_order, ordered=True)

season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Autumn", 10: "Autumn", 11: "Autumn"}
season_order = ["Winter", "Spring", "Summer", "Autumn"]

h["season"] = h["month"].map(season_map)

# remove the month filter (h = h[h["month"].isin(...)]) from the earlier code
profile = (
    h.groupby(["season", "day_type", "hour"], observed=True, as_index=False)["actual_consumption"].mean()
)

fig = px.line(
    profile, x="hour", y="actual_consumption", color="day_type",
    facet_col="season", category_orders={"season": season_order, "day_type": day_type_order},
    title="Average 24h load profile",
    labels={"actual_consumption": "MWh/h", "hour": "Hour of day"},
)
fig.update_xaxes(tickmode="linear", dtick=4)
fig.update_layout(title_x=0.5)
fig.show()



- **Observation** (hourly means, DST switch days excluded):
  - Winter weekday: night low ≈ 49k MWh/h (02:00), then a plateau of ≈ 70k with **two peaks at 11:00 and 18:00**, and only a slight dip in between (≈ 67k at 15:00).
  - Summer weekday: night low ≈ 42k (02:00), **a single peak at 11:00 (≈ 64k)**, then a gradual decline through the evening. No evening peak.
  - Weekend: the whole curve is lower and flatter. The morning ramp-up (06:00–09:00) is much weaker. In winter the weekend maximum is in the evening (18:00, ≈ 61k).
  - Weekday holidays are lower than regular weekends in every hour (winter ≈ 38–52k vs ≈ 46–61k; summer ≈ 35–49k vs ≈ 38–51k).
- **Interpretation**: The 06:00–09:00 ramp on weekdays reflects the start of work and industrial activity, which is absent on weekends. The winter evening peak is most likely lighting and heating when people are at home; it is missing in summer because it is still light. Weekday holidays being below weekends in absolute terms contrasts with the ratio view above (≈ 0.89 vs ≈ 0.87), because the ratio is relative to a weekly mean that the holiday itself pulls down. Holidays are also concentrated in winter and spring (Christmas, New Year, Easter), so the season mix is not identical to regular weekends.
- **Implication**: The hour-of-day shape depends on both day type and season, so an hourly model needs `hour × day_type` and `hour × season` interactions (or separate models). A daily model can ignore this. Spring and autumn are plotted but not discussed here.

### STL Decomposition - Daily Data

In [ ]:
from statsmodels.tsa.seasonal import MSTL

s = df_day_train.set_index("date_from")["actual_consumption"].asfreq("D")
res = MSTL(s, periods=(7, 365)).fit()
# res.seasonal has columns seasonal_7 and seasonal_365

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

comp = pd.DataFrame({
    "observed": res.observed, "trend": res.trend,
    "seasonal_7": res.seasonal["seasonal_7"], "seasonal_365": res.seasonal["seasonal_365"],
    "resid": res.resid,
})

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=list(comp.columns[:1]) + ["trend", "weekly seasonal", "yearly seasonal"])
for i, c in enumerate(["observed", "trend", "seasonal_7", "seasonal_365"], start=1):
    fig.add_trace(go.Scatter(x=comp.index, y=comp[c], mode="lines", showlegend=False), row=i, col=1)
fig.update_layout(height=700, title_text="MSTL decomposition", title_x=0.5)
fig.show()



# Residual: where are the anomalies?
fig = px.line(comp, y="resid", title="Residual")
fig.update_layout(height=700, title_x=0.5)
fig.show()


- **Observation**:
  - **Trend** (MSTL, periods = 7 and 365): ≈ 1.38M MWh/day (Jan 2019) → ≈ 1.33M (mid 2020) → back to a peak of ≈ 1.38M (Aug 2021) → a low of ≈ 1.25M (Apr 2023, about −9% from the 2021 peak) → ≈ 1.28M (end of 2024). The decline is not monotonic; most of it is one drop in 2022–2023.
  - **Weekly seasonal**: ≈ +100k on weekdays and ≈ −200k on weekends, stable over the six years (too dense to read at full range; zoom into a few weeks).
  - **Yearly seasonal**: winter high (≈ +200k), summer low, and a sharp drop around Christmas and New Year (≈ −200k to −300k).
  - **Residual**: ~39 days exceed ±3σ. Almost all are calendar events: Good Friday, Easter Monday, Ascension Day and Whit Monday (−130k to −230k) in every year. Weekend holidays (e.g. 2021-12-25/26) show **positive** residuals (+130k to +160k). The one unexplained cluster is **2022-04-01 to 2022-04-10** (+130k to +180k, 1.57M on 04-01).
- **Interpretation**: Christmas and New Year have fixed dates, so the yearly seasonal absorbs them. Easter-related holidays move every year, so they stay in the residual. Weekend holidays are positive because the model subtracts both the weekend effect and the holiday effect, but the two do not add up. The 2020 dip and the 2022–2023 drop are consistent with the first lockdown and the energy crisis, but this is not tested against temperature or price data. The MSTL trend is very smooth, so it blurs the exact timing of events.
- **Implication**: Keep `is_holiday`: it explains the large negative residuals on weekday holidays. Add the 4-class `day_type` (weekend × holiday) because weekend holidays do not behave like "weekend + holiday". Easter-specific features (days to Easter, Holy Saturday / Easter Sunday flags) and bridge-day flags are not justified yet: Holy Saturday and Easter Sunday are close to regular weekends (≈ 0.92 vs 0.90 and ≈ 0.83 vs 0.85 of the weekly mean), and only one bridge day (2021-05-14) shows up. Revisit only if model residuals still show the pattern. The 2022-04 cluster needs a check against the hourly data and a temperature series before deciding whether to keep or flag those days. The first and last half-year of the 365-day seasonal rest on few cycles and should not be over-interpreted.
- **TODO**: check 2022-04 hourly data; compare with temperature (e.g. DWD); FFT / power spectrum to confirm the 7- and 365-day periods.

## PV + wind vs consumption

`residual_load = actual_consumption - (PV + onshore wind + offshore wind)`, so the PV + wind total can be recovered from the existing columns. No extra download is needed for this section.

The target is `actual_consumption`, so `residual_load` is not a usable feature: it contains the target. The PV + wind values of the same day are also not available at prediction time. Only lagged values can be used, and the lag has to be at least the forecast horizon (1 day for a next-day forecast). All numbers below use the training set only.

In [ ]:
df_ren = df_day_train.sort_values("date_from").copy()
df_ren["date_from"] = pd.to_datetime(df_ren["date_from"])
df_ren["renewable_pv_wind"] = df_ren["actual_consumption"] - df_ren["residual_load"]
df_ren["season"] = df_ren["month"].map(season_map)  # season_map is defined in the intra-day section

print(df_ren[["actual_consumption", "residual_load", "renewable_pv_wind"]].describe().round(0))
print("corr(consumption, residual_load):", round(df_ren["actual_consumption"].corr(df_ren["residual_load"]), 3))
print("corr(consumption, renewable):    ", round(df_ren["actual_consumption"].corr(df_ren["renewable_pv_wind"]), 3))

fig = px.scatter(df_ren, x="renewable_pv_wind", y="actual_consumption", color="season",
                 color_discrete_sequence=["#0072B2", "#009E73", "#E69F00", "#D55E00"],
                 category_orders={"season": season_order},
                 opacity=0.6, title="Same-day PV + wind vs actual consumption (daily, train)",
                 labels={"renewable_pv_wind": "PV + wind (MWh/day)", "actual_consumption": "Actual consumption (MWh/day)"})
fig.update_layout(title_x=0.5)
fig.show()

print(df_ren.groupby("season")[["actual_consumption", "renewable_pv_wind"]].corr()
            .xs("actual_consumption", level=1)["renewable_pv_wind"].round(3))

- **Observation**: PV + wind averages ≈ 0.48M MWh/day (std ≈ 0.21M), against ≈ 1.33M for consumption. Same-day correlation with consumption is 0.10. Within each season it stays between -0.07 and 0.11. `residual_load` correlates 0.55 with consumption, not close to 1.
- **Interpretation**: PV + wind varies a lot from day to day, which weakens the link between `residual_load` and consumption. The weak overall correlation is not driven by one season.
- **Implication**: `residual_load` stays out of the feature set because it is built from the target, not because of collinearity. Same-day PV + wind shows almost no relationship even if it were available.

In [ ]:
# Remove the calendar effect: subtract the training mean of the same month x weekday
key = [df_ren["month"], df_ren["day_of_week_raw"]]
df_ren["cons_dev"] = df_ren["actual_consumption"] - df_ren.groupby(key)["actual_consumption"].transform("mean")
df_ren["ren_dev"] = df_ren["renewable_pv_wind"] - df_ren.groupby(key)["renewable_pv_wind"].transform("mean")

fig = px.scatter(df_ren, x="ren_dev", y="cons_dev", opacity=0.5,
                 color_discrete_sequence=["#0072B2"],
                 title="Deviation from month x weekday mean: PV + wind vs consumption",
                 labels={"ren_dev": "PV + wind deviation (MWh/day)", "cons_dev": "Consumption deviation (MWh/day)"})
fig.update_layout(title_x=0.5)
fig.show()

print("corr of deviations (same day):", round(df_ren["cons_dev"].corr(df_ren["ren_dev"]), 3))

- **Observation**: After subtracting the month × weekday mean, the same-day correlation between PV + wind and consumption is 0.02.
- **Interpretation**: The small raw correlation (0.10) is mostly shared seasonality, not a day-to-day link. Holidays are not removed here, so some noise remains.
- **Implication**: PV + wind does not explain deviations from the usual calendar pattern on the same day.

In [ ]:
# Lagged PV + wind vs consumption, and consumption vs its own lags for comparison
rows = []
for lag in [0, 1, 2, 7]:
    rows.append({
        "lag (days)": lag,
        "raw": df_ren["actual_consumption"].corr(df_ren["renewable_pv_wind"].shift(lag)),
        "calendar-adjusted": df_ren["cons_dev"].corr(df_ren["ren_dev"].shift(lag)),
    })
lag_table = pd.DataFrame(rows).set_index("lag (days)").round(3)
print("Consumption vs lagged PV + wind")
print(lag_table)

own = pd.DataFrame({
    "lag (days)": [1, 7],
    "raw": [df_ren["actual_consumption"].corr(df_ren["actual_consumption"].shift(l)) for l in [1, 7]],
    "calendar-adjusted": [df_ren["cons_dev"].corr(df_ren["cons_dev"].shift(l)) for l in [1, 7]],
}).set_index("lag (days)").round(3)
print("\nConsumption vs its own lag")
print(own)

# Partial correlation: lagged PV + wind vs consumption, controlling for consumption lag 1 and lag 7
X = pd.DataFrame({
    "y": df_ren["cons_dev"],
    "cons_l1": df_ren["cons_dev"].shift(1),
    "cons_l7": df_ren["cons_dev"].shift(7),
    "ren_l1": df_ren["ren_dev"].shift(1),
}).dropna()

def residualize(target, controls):
    A = np.c_[np.ones(len(controls)), controls.values]
    beta = np.linalg.lstsq(A, target.values, rcond=None)[0]
    return target.values - A @ beta

e_y = residualize(X["y"], X[["cons_l1", "cons_l7"]])
e_r = residualize(X["ren_l1"], X[["cons_l1", "cons_l7"]])
print("\npartial corr (PV + wind lag 1 | consumption lag 1, lag 7):", round(np.corrcoef(e_y, e_r)[0, 1], 3))

- **Observation**:
  - Lagged PV + wind vs consumption: raw correlation is 0.10 (lag 0), 0.07 (lag 1), 0.04 (lag 2), 0.02 (lag 7). After removing the month × weekday mean it is 0.02, -0.05, -0.11, -0.14.
  - Consumption vs its own lag: raw 0.67 (lag 1) and 0.86 (lag 7); calendar-adjusted 0.82 (lag 1) and 0.47 (lag 7).
  - Partial correlation of PV + wind (lag 1) with consumption, controlling for consumption lag 1 and lag 7: -0.11.
  - Removing only the month mean instead (same-day, lag 1, 2, 7): 0.01, -0.03, -0.07, -0.09; own lag 1 / lag 7 autocorrelation 0.55 / 0.80.
- **Interpretation**:
  - **Shared seasonality** means both series move with the time of year, not with each other. Monthly means (training set) are high in winter and low in late summer for both: consumption ≈ 1.46M MWh/day in Jan–Feb vs ≈ 1.23M in Aug; PV + wind ≈ 0.51–0.61M in Jan–Feb vs ≈ 0.40M in Aug–Sep. The 12 monthly means correlate at 0.72. A winter day therefore tends to have both higher consumption and higher PV + wind, which produces the small raw correlation even if one has no effect on the other.
  - The month mean is what removes this shared yearly pattern, which is why the same-day value falls from 0.10 to about 0.01–0.02. The weekday part is not about shared signal: PV + wind has no weekday pattern (≈ 0.48M every day), while consumption does (weekends ≈ 15% lower). Subtracting the weekday mean removes this consumption-only noise (std of `cons_dev` 140k → 86k), so the lag correlations are measured on a cleaner series. Without it, consumption lag 7 is 0.80, which only repeats the weekly cycle; with it, lag 1 stands out (0.82).
  - Both versions give the same picture: same-day about 0, lagged values slightly negative and growing with lag, small compared with the consumption autocorrelation. There is no direct mechanism by which yesterday's PV + wind output changes today's consumption, so the lagged values most likely pick up weather persistence. This is not tested.
  - Consumption itself is much better explained by its own recent values: lag 7 in raw terms (same weekday), lag 1 once the calendar effect is removed.
- **Implication**: PV + wind (actual or lagged) is not added as a feature. Weather effects are left to the temperature analysis. If PV + wind is revisited, day-ahead forecast values (SMARD "Prognostizierte Erzeugung") are available at prediction time and are the better source than lagged actuals. Calendar features and consumption lags (1 and 7) come first.